In [ ]:
# ONE-CELL — EXACT original CNN (no augmentation) + Leak-safe SBERT Fusion
# Paste this whole cell into Google Colab, use GPU runtime, upload kaggle.json when prompted.

# 0) Install & imports
!pip install -q kaggle sentence-transformers matplotlib seaborn joblib

import os, shutil, zipfile, re, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Input, Dense, Conv2D, MaxPooling2D, BatchNormalization,
                                     Flatten, Dropout, Concatenate)
from tensorflow.keras.utils import to_categorical
from sentence_transformers import SentenceTransformer
from sklearn.metrics import confusion_matrix, classification_report
import joblib

# reproducibility
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# 1) Upload kaggle.json
from google.colab import files
print('Upload your kaggle.json when prompted')
uploaded = files.upload()
if len(uploaded)==0:
    raise SystemExit('kaggle.json upload required')
name = list(uploaded.keys())[0]
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move(name, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ kaggle token installed')

# 2) Download dataset from Kaggle and extract
!kaggle datasets download -d subirbiswas19/skin-disease-dataset -q
if os.path.exists('skin-disease-dataset.zip'):
    with zipfile.ZipFile('skin-disease-dataset.zip','r') as z:
        z.extractall('/content')
else:
    raise FileNotFoundError('Dataset zip not found after download')

# 3) Paths & quick counts
def count_images(folder):
    total = 0
    for r,d,f in os.walk(folder):
        total += len([x for x in f if x.lower().endswith(('jpg','jpeg','png','webp'))])
    return total

data_root = "/content/skin-disease-datasaet"
train_root = os.path.join(data_root, "train_set")
test_root  = os.path.join(data_root, "test_set")

print("data_root:", data_root)
print("Train images:", count_images(train_root))
print("Test images :", count_images(test_root))

# 4) Detailed symptom lists (your original lists)
# 4) Detailed symptom lists (cleaned & accurate)
detailed_symptoms = {
    "bacellulitis":[
        "redness warmth swelling tenderness",
        "rapidly spreading redness usually one leg",
        "fever chills fatigue",
        "possible blistering on skin",
        "abscess formation",
        "peau d’orange orange peel texture"
    ],
    "baimpetigo":[
        "red sores or blisters that burst",
        "honey colored crusts",
        "itching",
        "mild discomfort",
        "lesions around nose or mouth",
        "spreads by scratching"
    ],
    "fuathletefoot":[
        "itching burning stinging between toes",
        "cracked scaly or peeling skin",
        "redness",
        "blisters on foot",
        "thick or discolored toenails if fungus spreads"
    ],
    "funailfungus":[
        "thick brittle crumbly nails",
        "yellow brown or white nail discoloration",
        "distorted nail shape",
        "pain when pressure applied to nail"
    ],
    "furingworm":[
        "ring shaped red scaly patches",
        "itching",
        "clear center with raised edges",
        "lesions expand outward",
        "well defined circular or oval borders"
    ],
    "pacutaneouslarvamigrans":[
        "itchy serpentine snake like tracks",
        "redness and inflammation",
        "tracks migrate daily as larvae move"
    ],
    "vichickenpox":[
        "red spots turning into fluid filled blisters then crusts",
        "itchy rash spreading from chest or back outward",
        "fever",
        "headache",
        "fatigue"
    ],
    "vishingles":[
        "burning or tingling pain before rash",
        "red patches forming clusters of blisters then crusts",
        "severe pain along one dermatome",
        "fever",
        "fatigue",
        "headache",
        "possible long term nerve pain postherpetic neuralgia"
    ]
}


# 5) Universal tokens and keyword map (for sanitization)
UNIVERSAL_SYMPTOMS = [
    "redness",
    "itching",
    "pain",
    "swelling",
    "blistering",
    "scaling",
    "circular rash",
    "burning",
    "fluid filled bumps",
    "skin peeling",
    "yellow thick nails",
    "crusting",
    "tingling pain",
    "fever rash",
    "track-like rash"
]

keyword_map = {
    # Redness / inflammation
    "red": "redness", "redness": "redness", "reddish": "redness",
    "inflam": "redness", "warm": "redness", "hot": "redness",
    "rash": "redness", "erupt": "redness",

    # Itching
    "itch": "itching", "itchy": "itching", "scratch": "itching",
    "prurit": "itching",

    # Pain / tenderness
    "pain": "pain", "painful": "pain", "tender": "pain",
    "sore": "pain", "hurts": "pain",

    # Swelling
    "swel": "swelling", "swelling": "swelling", "puffy": "swelling",
    "raised": "swelling",

    # Blistering / vesicles
    "blister": "blistering", "bubble": "blistering",
    "vesicle": "blistering", "water filled": "blistering",
    "fluid": "fluid filled bumps", "pus": "fluid filled bumps",

    # Scaling / dryness
    "scal": "scaling", "flake": "scaling", "dry": "scaling",
    "peel": "skin peeling", "crack": "skin peeling",

    # Nails
    "nail": "yellow thick nails", "yellow": "yellow thick nails",
    "thick": "yellow thick nails", "crumb": "yellow thick nails",

    # Ring-shaped lesions
    "ring": "circular rash", "circle": "circular rash",
    "circular": "circular rash", "border": "circular rash",

    # Burning / tingling (shingles)
    "burn": "burning", "burning": "burning",
    "tingl": "tingling pain", "shooting": "tingling pain",

    # Fever-related
    "fever": "fever rash", "chills": "fever rash",

    # Track-like CLM patterns
    "track": "track-like rash", "line": "track-like rash",
    "serpentine": "track-like rash", "snake": "track-like rash"
}



def map_to_universal(phrase):
    phrase_l = phrase.lower()
    for k,v in keyword_map.items():
        if k in phrase_l:
            return v
    return random.choice(UNIVERSAL_SYMPTOMS)

# 6) Build sym_df mapping from dataset files -> sanitized universal token
rows = []
for root in [train_root, test_root]:
    for folder in sorted(os.listdir(root)):
        folder_path = os.path.join(root, folder)
        if not os.path.isdir(folder_path):
            continue
        key = re.sub(r'[^a-z0-9]+','', folder.lower())
        detailed_list = detailed_symptoms.get(key, [])
        imgs = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('jpg','jpeg','png','webp'))])
        for i,fname in enumerate(imgs):
            if detailed_list:
                phrase = detailed_list[i % len(detailed_list)]
                uni = map_to_universal(phrase)
            else:
                uni = random.choice(UNIVERSAL_SYMPTOMS)
            sym_text = uni
            rel = os.path.join(folder, fname)
            rows.append([rel, sym_text])

sym_df = pd.DataFrame(rows, columns=['relpath','symptoms'])
sym_df.to_csv('symptoms.csv', index=False)
print('symptoms.csv saved, rows=', len(sym_df))

# 7) Image generators — NO augmentation (exact original style)
IMG_SIZE = (150,150)
BATCH = 32

data_gen = ImageDataGenerator(rescale=1./255)

train_generator = data_gen.flow_from_directory(
    train_root, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=True, seed=SEED)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    test_root, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)

NUM_CLASSES = train_generator.num_classes
CLASS_NAMES = list(train_generator.class_indices.keys())
print('Classes:', CLASS_NAMES)

# 8) Build EXACT original CNN (32,64,128,128,512,512) with your dropout style
model = Sequential([
    Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    Conv2D(32, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(512, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Conv2D(512, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2,2),
    Flatten(),
    Dense(512, activation='relu', name='image_features'),
    Dropout(0.6),
    Dense(128, activation='relu'),
    Dropout(0.6),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

# 9) Train CNN EXACTLY like your standalone (no callbacks, full epochs)
EPOCHS_CNN = 25
history = model.fit(train_generator, epochs=EPOCHS_CNN, validation_data=val_gen, verbose=1)

# Plot training curves
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.plot(history.history.get('accuracy',[]), label='train'); plt.plot(history.history.get('val_accuracy',[]), label='val'); plt.legend(); plt.title('Accuracy')
plt.subplot(1,2,2); plt.plot(history.history.get('loss',[]), label='train'); plt.plot(history.history.get('val_loss',[]), label='val'); plt.legend(); plt.title('Loss')
plt.show()

cnn_test_loss, cnn_test_acc = model.evaluate(val_gen, verbose=0)
print(f'CNN baseline accuracy: {cnn_test_acc*100:.2f}%')

# 10) Build feature extractor (image_features)
_ = model.predict(np.zeros((1,IMG_SIZE[0],IMG_SIZE[1],3)))
feature_extractor = Model(inputs=model.layers[0].input, outputs=model.get_layer('image_features').output)
print('Feature extractor ready')

# 11) Ordered generators for embeddings (no shuffle)
ordered_train = ImageDataGenerator(rescale=1./255).flow_from_directory(train_root, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)
ordered_val   = ImageDataGenerator(rescale=1./255).flow_from_directory(test_root, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)

train_steps = math.ceil(ordered_train.samples / BATCH)
val_steps   = math.ceil(ordered_val.samples / BATCH)

# 12) Extract image embeddings
train_img_emb = feature_extractor.predict(ordered_train, steps=train_steps, verbose=1)
val_img_emb   = feature_extractor.predict(ordered_val, steps=val_steps, verbose=1)
print('Image embeddings shapes:', train_img_emb.shape, val_img_emb.shape)

# 13) Build text inputs for SBERT — sanitized universal tokens
sym_map = dict(zip(sym_df.relpath, sym_df.symptoms))
train_texts = [sym_map.get(fp, random.choice(UNIVERSAL_SYMPTOMS)) for fp in ordered_train.filenames]
val_texts   = [sym_map.get(fp, random.choice(UNIVERSAL_SYMPTOMS)) for fp in ordered_val.filenames]

# Add a small fraction of generic "unspecified" during training so model can't fully rely on text
def add_generic_dropout(texts, drop_frac=0.10):
    out = []
    for t in texts:
        if random.random() < drop_frac:
            out.append("unspecified skin complaint")
        else:
            out.append(t)
    return out

train_texts_aug = add_generic_dropout(train_texts, drop_frac=0.10)

# 14) Text embeddings with SBERT
text_model = SentenceTransformer('all-MiniLM-L6-v2')
train_text_emb = text_model.encode(train_texts_aug, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
val_text_emb   = text_model.encode(val_texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
print('Text embeddings shapes:', train_text_emb.shape, val_text_emb.shape)

# 15) Prepare labels
y_train = to_categorical(ordered_train.labels, num_classes=NUM_CLASSES)
y_val   = to_categorical(ordered_val.labels, num_classes=NUM_CLASSES)

# 16) Fusion model (project SBERT -> image dim, concat)
img_dim = train_img_emb.shape[1]
txt_dim = train_text_emb.shape[1]

img_in = Input(shape=(img_dim,), name='img_in')
txt_in = Input(shape=(txt_dim,), name='txt_in')

txt_proj = Dense(img_dim, activation='relu', name='txt_proj')(txt_in)
txt_proj = BatchNormalization()(txt_proj)
txt_proj = Dropout(0.2)(txt_proj)

fusion = Concatenate(name='fusion')([img_in, txt_proj])
x = Dense(512, activation='relu')(fusion); x = BatchNormalization()(x); x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x); x = BatchNormalization()(x); x = Dropout(0.2)(x)
x = Dense(128, activation='relu')(x); x = Dropout(0.15)(x)
out = Dense(NUM_CLASSES, activation='softmax')(x)

fusion_model = Model([img_in, txt_in], out, name='fusion_model')
fusion_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4), loss='categorical_crossentropy', metrics=['accuracy'])
fusion_model.summary()

# 17) Train fusion (on embeddings) — use callbacks here (small model)
fusion_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint('/content/best_fusion.h5', monitor='val_accuracy', save_best_only=True, mode='max')
]

history_f = fusion_model.fit([train_img_emb, train_text_emb], y_train,
                             validation_data=([val_img_emb, val_text_emb], y_val),
                             epochs=30, batch_size=16, callbacks=fusion_callbacks, verbose=1)

# 18) Evaluate fusion
test_loss, test_acc = fusion_model.evaluate([val_img_emb, val_text_emb], y_val, verbose=0)
print(f'Fusion test accuracy: {test_acc*100:.2f}% (CNN baseline {cnn_test_acc*100:.2f}%)')

# 19) Confusion matrix & classification report
pred_probs = fusion_model.predict([val_img_emb, val_text_emb])
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = np.argmax(y_val, axis=1)
class_names = list(ordered_train.class_indices.keys())

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10,8)); sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names); plt.title('Confusion Matrix'); plt.show()

print('\nClassification Report:\n')
print(classification_report(true_labels, pred_labels, target_names=class_names, digits=4))

# 20) Save artifacts
fusion_model.save('/content/fusion_model.h5')
model.save('/content/cnn_image_model.h5')
feature_extractor.save('/content/feature_extractor.h5')
joblib.dump(class_names, '/content/class_names.pkl')
joblib.dump(sym_map, '/content/sym_map.pkl')

print('Saved models & artifacts to /content')

# 21) Inference helper (sanitizes input)
from tensorflow.keras.preprocessing import image as kimg

def predict_multimodal(image_path, symptoms_text):
    """
    image_path: local path to image file
    symptoms_text: free-text symptom description provided by user
    """
    img = kimg.load_img(image_path, target_size=IMG_SIZE)
    arr = kimg.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    img_emb = feature_extractor.predict(arr, verbose=0)

    txt = symptoms_text.lower()
    mapped = None
    for k,v in keyword_map.items():
        if k in txt:
            mapped = v
            break
    if mapped is None:
        mapped = "unspecified skin complaint"
    txt_emb = text_model.encode([mapped], convert_to_numpy=True)

    probs = fusion_model.predict([img_emb, txt_emb], verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), probs

print('\nInference ready. Example:')
print("predict_multimodal('/content/skin-disease-datasaet/test_set/BA- cellulitis/BA- cellulitis (1).jpeg', 'redness and swelling')")

# END ONE-CELL
